# BanglaBERT: NER (Multi-word BIO) + Sentiment Analysis + Zero-Shot Classification

This notebook fine-tunes **BanglaBERT** (`csebuetnlp/banglabert`) for:
1. **Named Entity Recognition (NER)** – extracting product entities using BIO tagging
2. **Sentiment Analysis (SA)** – classifying reviews as Negative / Neutral / Positive
3. **Zero-Shot Classification** – classifying reviews without task-specific training

**Training data:** Gemini 3.5-Flash Synthetic Dataset  
**Test data:** Test.csv  
**Model:** `csebuetnlp/banglabert`

## A. Install Required Libraries
Install Hugging Face `transformers`, `datasets`, `evaluate`, `seqeval` (for token-level NER metrics),
`torch` as the deep-learning backend, `sentencepiece` for sub-word tokenization,
and `accelerate>=1.1.0` (required by `Trainer` in newer transformers).

In [5]:
# -*- coding: utf-8 -*-
!pip install datasets tokenizers seqeval -q
!pip install transformers
!pip install torch
!pip install sentencepiece
!pip install evaluate
!pip install 'accelerate>=1.1.0' -q

ERROR: Invalid requirement: "'accelerate": Expected package name at the start of dependency specifier
    'accelerate
    ^


## B. Import Libraries
Import standard data-science and NLP libraries. We disable Weights & Biases logging
via the `WANDB_DISABLED` environment variable so training runs stay local.

In [6]:
import os
os.environ["WANDB_DISABLED"] = "true"

import re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, classification_report
)
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForTokenClassification,
    AutoModelForSequenceClassification, TrainingArguments, Trainer,
    DataCollatorForTokenClassification, DataCollatorWithPadding,
    pipeline
)
import evaluate

## C. Load Datasets
Load the **Gemini 3.5-Flash Synthetic Dataset** as the training set and the **Test** dataset
from the `Synthetic Datasets/` folder. Both CSVs contain columns: `id`, `review`, `sentiment`,
and `bio_tagged_review`.

In [8]:
TRAIN_CSV = '../Synthetic Datasets/Gemini 3.5-Flash_Synthetic_Dataset.csv'
TEST_CSV  = '../Synthetic Datasets/Test.csv'

df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f"Training samples: {len(df_train)}")
print(f"Test samples:     {len(df_test)}")
df_train.head()

Training samples: 3000
Test samples:     30


,id,review,sentiment,bio_tagged_review
0,1,জঘন্য ফেস ওয়াশ দিয়েছে। খুবই হতাশ।,0,জঘন্য ফেস-B ওয়াশ-I দিয়েছে। খুবই হতাশ।
1,2,১০০% অরিজিনাল ফেস ওয়াশ পেয়েছি। খুবই পছন্দ হয়েছে।,2,১০০% অরিজিনাল ফেস-B ওয়াশ-I পেয়েছি। খুবই পছন্দ...
2,3,ফেস ওয়াশ টা দাম অনুযায়ী ঠিক আছে। আরেকটু ভালো ...,1,ফেস-B ওয়াশ-I টা দাম অনুযায়ী ঠিক আছে। আরেকটু ভ...
3,4,"ফেস ওয়াশ নিয়ে নতুন কি আর বলব, ভালো তো বটেই।",2,"ফেস-B ওয়াশ-I নিয়ে নতুন কি আর বলব, ভালো তো বটেই।"
4,5,অসাধারণ ফেস ওয়াশ পেয়েছি। অনেক দিন পর ভালো জিন...,2,অসাধারণ ফেস-B ওয়াশ-I পেয়েছি। অনেক দিন পর ভালো...


## D. Parse BIO Tags into Tokens & NER Labels
Each `bio_tagged_review` string contains tokens annotated as `token-B` or `token-I`.
We parse them into parallel lists of **tokens** and **BIO labels** (`B-Product`, `I-Product`, `O`).
For the test set, if `bio_tagged_review` is missing we fall back to whitespace tokenization.

In [9]:
def parse_bio_tags(bio_text):
    tokens, labels = [], []
    for part in bio_text.split():
        match = re.match(r'^(.*?)(?:[,\s।]*)-(B|I)$', part)
        if match:
            token, tag = match.group(1), match.group(2)
            labels.append('B-Product' if tag == 'B' else 'I-Product')
        else:
            token = part
            labels.append('O')
        tokens.append(token)
    return tokens, labels

# Parse training set
df_train['tokens']   = df_train['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[0])
df_train['ner_tags'] = df_train['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[1])

# Parse test set (fall back to plain split if no BIO column)
if 'bio_tagged_review' in df_test.columns:
    df_test['tokens']   = df_test['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[0])
    df_test['ner_tags'] = df_test['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[1])
else:
    df_test['tokens'] = df_test['review'].apply(lambda x: x.split())

print(df_train[['review', 'tokens', 'ner_tags']].iloc[0])

review             জঘন্য ফেস ওয়াশ দিয়েছে। খুবই হতাশ।
tokens      [জঘন্য, ফেস, ওয়াশ, দিয়েছে।, খুবই, হতাশ।]
ner_tags           [O, B-Product, I-Product, O, O, O]
Name: 0, dtype: object


## E. Define Label Mappings
Create bidirectional mappings between label strings and integer IDs.
Our NER scheme has three labels: `O` (non-entity), `B-Product` (beginning of product),
and `I-Product` (inside product).

In [10]:
label_list = ['O', 'B-Product', 'I-Product']
id2label   = {i: label for i, label in enumerate(label_list)}
label2id   = {label: i for i, label in enumerate(label_list)}
num_labels = len(label_list)

print(f"Labels ({num_labels}): {label_list}")
print(f"label2id: {label2id}")

Labels (3): ['O', 'B-Product', 'I-Product']
label2id: {'O': 0, 'B-Product': 1, 'I-Product': 2}


## F. Split Training Data (80/20)
Create a stratified 80/20 train/validation split so we can monitor overfitting during training.
Stratification on `sentiment` ensures the class distribution stays consistent across splits.

In [11]:
X = df_train[['id', 'review', 'tokens', 'ner_tags']]
y = df_train['sentiment']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.reset_index(drop=True)
X_val   = X_val.reset_index(drop=True)

print(f"Train size: {len(X_train)},  Validation size: {len(X_val)}")

Train size: 2400,  Validation size: 600


## G. NER Tokenizer & Label Alignment (BanglaBERT)
Load the **BanglaBERT** tokenizer (`csebuetnlp/banglabert`). Because sub-word tokenization
splits a single word into multiple tokens, we align NER labels using `word_ids()`:
only the **first sub-token** of each word receives the true label; the rest get `-100`
(ignored during loss computation).

In [12]:
BANGLABERT = "csebuetnlp/banglabert"

tokenizer_ner = AutoTokenizer.from_pretrained(BANGLABERT)

def tokenize_and_align_labels(examples, tokenizer, label2id):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        padding=False
    )
    all_labels = []
    for i, label_seq in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label_seq[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        all_labels.append(label_ids)
    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs

def create_ner_dataset(df, tokenizer, label2id):
    tokenized = tokenize_and_align_labels(
        {'tokens': df['tokens'].tolist(), 'ner_tags': df['ner_tags'].tolist()},
        tokenizer, label2id
    )
    return Dataset.from_dict({
        'input_ids':      tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels':         tokenized['labels'],
        'tokens':         df['tokens'].tolist(),
        'ner_tags':       df['ner_tags'].tolist(),
        'id':             df['id'].tolist() if 'id' in df else list(range(len(df)))
    })

train_dataset_ner = create_ner_dataset(X_train, tokenizer_ner, label2id)
val_dataset_ner   = create_ner_dataset(X_val,   tokenizer_ner, label2id)

if 'tokens' in df_test.columns and 'ner_tags' in df_test.columns:
    test_dataset_ner = create_ner_dataset(df_test, tokenizer_ner, label2id)
else:
    test_dataset_ner = None

print(f"NER datasets created -- train: {len(train_dataset_ner)}, val: {len(val_dataset_ner)}")

NER datasets created -- train: 2400, val: 600


## H. Train the NER Model (BanglaBERT Token Classification)
Combine the train + validation splits into one full training set, then fine-tune
**BanglaBERT** for token classification (3 epochs). We evaluate on the held-out
validation set using the `seqeval` metric and save the best checkpoint.

In [13]:
print("=== Training NER model on full training data ===\n")

# Merge train + val for final training
full_train = pd.concat([X_train, X_val], ignore_index=True)
full_train['sentiment'] = pd.concat([y_train, y_val], ignore_index=True)
full_train_ds = create_ner_dataset(full_train, tokenizer_ner, label2id)

# Build model from BanglaBERT pretrained weights
config = AutoConfig.from_pretrained(BANGLABERT)
config.num_labels = num_labels
config.id2label   = id2label
config.label2id   = label2id
model_ner = AutoModelForTokenClassification.from_pretrained(
    BANGLABERT, config=config
)

args_ner = TrainingArguments(
    output_dir="ner_banglabert_final",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
)

data_collator_ner = DataCollatorForTokenClassification(tokenizer_ner)
metric_ner = evaluate.load("seqeval")

def compute_metrics_ner(eval_preds):
    pred_logits, labels = eval_preds
    pred_logits = np.argmax(pred_logits, axis=2)
    predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(pred_logits, labels)
    ]
    references = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(pred_logits, labels)
    ]
    results = metric_ner.compute(predictions=predictions, references=references)
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

trainer_ner = Trainer(
    model=model_ner,
    args=args_ner,
    train_dataset=full_train_ds,
    eval_dataset=val_dataset_ner,
    data_collator=data_collator_ner,
    processing_class=tokenizer_ner,
    compute_metrics=compute_metrics_ner
)

trainer_ner.train()

# Evaluate on test set if available
if test_dataset_ner is not None:
    test_results_ner = trainer_ner.evaluate(test_dataset_ner)
    print("\n=== NER Test Results ===")
    print(test_results_ner)

model_ner.save_pretrained("ner_banglabert_model_final")
tokenizer_ner.save_pretrained("ner_banglabert_tokenizer")
print("\nNER model saved.")

=== Training NER model on full training data ===



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForTokenClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.bias                                   | MISSING    | 
classifier.weight                                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\ProgramData\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no acc

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.004537,0.001272,1.000000,1.000000,1.000000,1.000000
2,0.001552,0.001400,0.998314,1.000000,0.999156,0.999798


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\ProgramData\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SafetensorError: Error while serializing: I/O error: There is not enough space on the disk. (os error 112)

## I. Sentiment Analysis – Tokenize & Prepare Datasets (BanglaBERT)
Load a separate **BanglaBERT** tokenizer for sequence classification.
Tokenize the raw review text and create `Dataset` objects for train, validation, and test splits.

In [ ]:
sa_tokenizer = AutoTokenizer.from_pretrained(BANGLABERT)

def tokenize_sa(reviews, tokenizer):
    return tokenizer(reviews, truncation=True, padding=False)

# Prepare text & labels for each split
X_train_sa      = X_train['review'].tolist()
y_train_sa      = y_train.tolist()
X_val_sa        = X_val['review'].tolist()
y_val_sa        = y_val.tolist()
full_train_sa   = full_train['review'].tolist()
full_train_sa_y = full_train['sentiment'].tolist()

# Tokenize
train_enc_sa      = tokenize_sa(X_train_sa,    sa_tokenizer)
val_enc_sa        = tokenize_sa(X_val_sa,      sa_tokenizer)
full_train_enc_sa = tokenize_sa(full_train_sa,  sa_tokenizer)

# Build HuggingFace Datasets
train_sa_dataset = Dataset.from_dict({
    'input_ids':      train_enc_sa['input_ids'],
    'attention_mask': train_enc_sa['attention_mask'],
    'labels':         y_train_sa,
})
val_sa_dataset = Dataset.from_dict({
    'input_ids':      val_enc_sa['input_ids'],
    'attention_mask': val_enc_sa['attention_mask'],
    'labels':         y_val_sa,
})
full_train_sa_dataset = Dataset.from_dict({
    'input_ids':      full_train_enc_sa['input_ids'],
    'attention_mask': full_train_enc_sa['attention_mask'],
    'labels':         full_train_sa_y,
})

print(f"SA datasets created -- train: {len(train_sa_dataset)}, val: {len(val_sa_dataset)}")

## J. Train the Sentiment Analysis Model (BanglaBERT Sequence Classification)
Fine-tune **BanglaBERT** for 3-class sentiment classification (Negative / Neutral / Positive)
on the full training set. The best checkpoint (by validation accuracy) is kept.

In [ ]:
print("=== Training final Sentiment Analysis model ===\n")

model_sa = AutoModelForSequenceClassification.from_pretrained(
    BANGLABERT,
    num_labels=3,
    id2label={0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"},
    label2id={"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}
)

args_sa = TrainingArguments(
    output_dir="sa_banglabert_final",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
)

data_collator_sa = DataCollatorWithPadding(sa_tokenizer)

def compute_metrics_sa(eval_pred):
    preds  = eval_pred.predictions.argmax(-1)
    labels = eval_pred.label_ids
    return {
        'precision': precision_score(labels, preds, average='weighted'),
        'recall':    recall_score(labels, preds, average='weighted'),
        'f1':        f1_score(labels, preds, average='weighted'),
        'accuracy':  accuracy_score(labels, preds)
    }

trainer_sa = Trainer(
    model=model_sa,
    args=args_sa,
    train_dataset=full_train_sa_dataset,
    eval_dataset=val_sa_dataset,
    processing_class=sa_tokenizer,
    data_collator=data_collator_sa,
    compute_metrics=compute_metrics_sa
)

trainer_sa.train()
print("\nSA model training complete.")

## K. Evaluate Sentiment Model on Test Set
Tokenize the test-set reviews and run evaluation. If the test CSV lacks a `sentiment`
column, dummy labels are used (accuracy will be meaningless but the pipeline won't crash).

In [ ]:
test_reviews  = df_test['review'].tolist()
test_enc_sa   = tokenize_sa(test_reviews, sa_tokenizer)

if 'sentiment' in df_test.columns:
    test_labels = df_test['sentiment'].tolist()
else:
    test_labels = [0] * len(df_test)
    print("Warning: No sentiment column in test set - using dummy labels.")

test_sa_dataset = Dataset.from_dict({
    'input_ids':      test_enc_sa['input_ids'],
    'attention_mask': test_enc_sa['attention_mask'],
    'labels':         test_labels
})

test_results_sa = trainer_sa.evaluate(test_sa_dataset)
print("\n=== SA Test Results ===")
print(test_results_sa)

model_sa.save_pretrained("sa_banglabert_model_final")
sa_tokenizer.save_pretrained("sa_banglabert_tokenizer")
print("SA model saved.")

## L. Inference Example
Demonstrate end-to-end inference on a sample Bangla review:
1. **NER pipeline** – extract product entities from the text.
2. **Sentiment classification** – predict the overall sentiment (Negative / Neutral / Positive).

In [ ]:
# --- NER inference ---
ner_pipe = pipeline(
    "ner",
    model="ner_banglabert_model_final",
    tokenizer="ner_banglabert_tokenizer",
    aggregation_strategy="simple"
)

sample_review = "ফেস ওয়াশ টা অনেক সুন্দর, অসংখ্য ধন্যবাদ ডেলিভারি খুব দ্রুত দেয়ার জন্য।"
print("Sample review:", sample_review)
print("NER results:", ner_pipe(sample_review))

# --- Sentiment inference ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_sa.to(device)

enc = sa_tokenizer([sample_review], padding=True, truncation=True, return_tensors="pt")
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    logits = model_sa(**enc).logits
    pred = logits.argmax(dim=1).item()

sentiment_map = {0: "Negative", 1: "Neutral", 2: "Positive"}
print(f"\nPredicted sentiment: {sentiment_map[pred]}")

## M. Zero-Shot Sentiment Classification
Run **zero-shot classification** using a multilingual NLI model
(`MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual`) to classify Bangla reviews
into Negative / Neutral / Positive **without any task-specific fine-tuning**.
This provides a baseline comparison against the fine-tuned models.

In [ ]:
# Load zero-shot pipeline (multilingual NLI-based)
zs_model_name = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual"
print(f"Loading zero-shot model: {zs_model_name}\n")

zs_pipe = pipeline(
    "zero-shot-classification",
    model=zs_model_name,
    tokenizer=zs_model_name,
)

# Candidate labels for sentiment
candidate_labels = ["negative", "neutral", "positive"]

# Run on test set
zs_results = []
for idx, review in enumerate(df_test['review'].tolist()):
    result = zs_pipe(review, candidate_labels=candidate_labels)
    predicted_label = result['labels'][0]
    predicted_score = result['scores'][0]
    zs_results.append({
        'id': idx + 1,
        'review': review[:80],
        'predicted_sentiment': predicted_label,
        'confidence': round(predicted_score, 4),
        'all_labels': result['labels'],
        'all_scores': [round(s, 4) for s in result['scores']]
    })
    if (idx + 1) % 10 == 0:
        print(f"  Processed {idx + 1}/{len(df_test)} reviews")

df_zs = pd.DataFrame(zs_results)
print(f"\nZero-shot classification complete for {len(df_zs)} reviews.")
df_zs.head(10)

### M.1 Zero-Shot Evaluation Metrics
Compare zero-shot predictions against ground-truth labels (if available)
and compute standard classification metrics.

In [ ]:
# Map zero-shot labels to integer IDs for comparison
zs_label_map = {"negative": 0, "neutral": 1, "positive": 2}
df_zs['pred_id'] = df_zs['predicted_sentiment'].map(zs_label_map)

if 'sentiment' in df_test.columns:
    true_labels = df_test['sentiment'].tolist()
    zs_preds    = df_zs['pred_id'].tolist()

    zs_accuracy  = accuracy_score(true_labels, zs_preds)
    zs_precision = precision_score(true_labels, zs_preds, average='weighted')
    zs_recall    = recall_score(true_labels, zs_preds, average='weighted')
    zs_f1        = f1_score(true_labels, zs_preds, average='weighted')

    print("=== Zero-Shot Classification Metrics ===")
    print(f"  Accuracy:  {zs_accuracy:.4f}")
    print(f"  Precision: {zs_precision:.4f}")
    print(f"  Recall:    {zs_recall:.4f}")
    print(f"  F1 Score:  {zs_f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(true_labels, zs_preds,
          target_names=['Negative', 'Neutral', 'Positive']))

    # Confusion matrix
    cm = confusion_matrix(true_labels, zs_preds)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Neutral', 'Positive'],
                yticklabels=['Negative', 'Neutral', 'Positive'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Zero-Shot Confusion Matrix (BanglaBERT)')
    plt.tight_layout()
    plt.show()
else:
    print("No ground-truth sentiment labels available for zero-shot evaluation.")
    zs_accuracy = zs_precision = zs_recall = zs_f1 = None

## N. Save All Reports to `output_txt/`
Compile NER test results, SA test results, zero-shot metrics, and a
full summary into a text report and save it under `output_txt/`.

In [ ]:
import os
from datetime import datetime

os.makedirs("output_txt", exist_ok=True)

report_file = f"output_txt/BanglaBERT_GEMINI_FLASH_EXTENDED_Report.txt"

with open(report_file, "w", encoding="utf-8") as f:
    f.write("=" * 70 + "\n")
    f.write(f"  BanglaBERT – GEMINI FLASH EXTENDED – Full Report\n")
    f.write(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 70 + "\n\n")

    f.write(f"Model: {model_id}\n")
    f.write(f"Training samples: {len(df_train)}\n")
    f.write(f"Test samples:     {len(df_test)}\n\n")

    # --- NER Results ---
    f.write("-" * 50 + "\n")
    f.write("NER Test Results\n")
    f.write("-" * 50 + "\n")
    if test_dataset_ner is not None:
        for k, v in test_results_ner.items():
            f.write(f"  {k}: {v}\n")
    else:
        f.write("  No test NER dataset available.\n")
    f.write("\n")

    # --- SA Results ---
    f.write("-" * 50 + "\n")
    f.write("Sentiment Analysis Test Results\n")
    f.write("-" * 50 + "\n")
    for k, v in test_results_sa.items():
        f.write(f"  {k}: {v}\n")
    f.write("\n")

    # --- Zero-Shot Results ---
    f.write("-" * 50 + "\n")
    f.write("Zero-Shot Classification Results\n")
    f.write("-" * 50 + "\n")
    f.write(f"  Zero-shot model: {zs_model_name}\n")
    if zs_accuracy is not None:
        f.write(f"  Accuracy:  {zs_accuracy:.4f}\n")
        f.write(f"  Precision: {zs_precision:.4f}\n")
        f.write(f"  Recall:    {zs_recall:.4f}\n")
        f.write(f"  F1 Score:  {zs_f1:.4f}\n")
    else:
        f.write("  No ground-truth labels for evaluation.\n")
    f.write("\n")

    # --- Zero-shot per-sample predictions ---
    f.write("-" * 50 + "\n")
    f.write("Zero-Shot Per-Sample Predictions\n")
    f.write("-" * 50 + "\n")
    for _, row in df_zs.iterrows():
        f.write(f"  [{row['id']:>3}] {row['predicted_sentiment']:>8} "
                f"(conf={row['confidence']:.4f}) | {row['review']}\n")
    f.write("\n")

    f.write("=" * 70 + "\n")
    f.write("End of Report\n")
    f.write("=" * 70 + "\n")

print(f"Report saved to: {report_file}")
print("\nAll tasks completed successfully!")